In [33]:
import pandas as pd
import os
import ast
import re

In [44]:
list_name = os.listdir('./')
study_group = 'DIET'

In [35]:
list_name

['demo_script.ipynb',
 'DIET_nhanes_codebook_COMPLETAS.csv',
 'DIET_nhanes_codebook_DR1IFF_C_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_D_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_E_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_F_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_G_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_H_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_I_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_J_resultados.csv',
 'DIET_nhanes_codebook_DR1IFF_L_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_C_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_D_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_E_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_F_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_G_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_H_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_I_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_J_resultados.csv',
 'DIET_nhanes_codebook_DR1TOT_L_resultados.csv',
 'DIET_nhanes_codebook_DR2IFF_C_resultados.csv',
 'DIET_n

In [37]:

name_tables = [palavra.split("DIET_nhanes_codebook_")[1] for palavra in list_name if palavra.endswith('.csv')]
name_tables = [palavra.split("_") for palavra in name_tables if palavra.endswith('resultados.csv')]

nome_unique = []
for lista in name_tables:
    if len(lista[0])>1:
        nome_unique.append(lista[0])
    elif len(lista[0])==1:
        nome_unique.append(lista[1])
        
nome_unique = list(set(nome_unique))

In [41]:
def process_string_to_dict(string):
    string = string.split('; Count')[0]
    # Substituir os `;` por vírgulas e remover espaços extras
    formatted_string = re.sub(r";", ",", string)
    formatted_string = re.sub(r"=", ":", formatted_string)
    # Adicionar chaves externas para transformá-la em um dicionário
    formatted_string = "{" + formatted_string + "}"
    # Substituir chaves como "Code.or.Value" por "Code" e "Value.Description" por "Description"
    formatted_string = re.sub(r"Code\.or\.Value", '"Code"', formatted_string)
    formatted_string = re.sub(r"Value\.Description", '"Description"', formatted_string)

    return eval(formatted_string)

In [ ]:
for nome in nome_unique:
    lista_filtrada = [item for item in list_name if nome in item]
    # print()
    # print("********************************")
    # print(lista_filtrada)
    
    df_now_total = pd.DataFrame()
    for item in lista_filtrada:
        df_now = pd.read_csv(item)
        df_now_total = pd.concat([df_now_total, df_now])
        
        
    list_Variable_Name = df_now_total['Variable_Name'].unique()
    df_codebook_final = pd.DataFrame()
    for variable in list_Variable_Name:
        df_now = df_now_total[df_now_total['Variable_Name']==variable]
        list_english = list(df_now['English_Text'].unique())
        list_Target = list(df_now['Target'].unique())
        list_English_Instructions = list(df_now['English_Instructions'].unique())
        df_now['English_Text'] = str(list_english)
        df_now['Target'] = str(list_Target)
        df_now['English_Instructions'] = str(list_English_Instructions)
        df_codebook_final = pd.concat([df_codebook_final, df_now])
    
    #display(df_codebook_final)

    rows = []
    for _, row in df_codebook_final.iterrows():
        # Ignorar linhas onde Tabela_Valores é NaN
        if pd.isna(row['Tabela_Valores']):
            rows.append({
                    "Column": row["Variable_Name"],
                    "Code": None,
                    "Label": None,
                    "English_Text": row["English_Text"],
                    "Target": row["Target"],
                    "English_Instructions": row["English_Instructions"]
                })
            continue
        
        # Extrair os dados de Code e Value.Description
        valores = row['Tabela_Valores']
        
        try:
            # Usando ast.literal_eval para transformar a string em dicionário
            parsed_values = process_string_to_dict(valores)
            codes = parsed_values["Code"]
            codes = [str(item).title() for item in codes]
            descriptions = parsed_values["Description"]
            descriptions = [str(item).title() for item in descriptions]
            # Criar uma nova linha para cada combinação de Code e Description
            for code, description in zip(codes, descriptions):
                rows.append({
                    "Column": row["Variable_Name"],
                    "Code": code,
                    "Label": description,
                    "English_Text": row["English_Text"],
                    "Target": row["Target"],
                    "English_Instructions": row["English_Instructions"]
                })
        except Exception as e:
            print(f"Erro ao processar a linha: {row['Tabela_Valores']}")
            print(e)

    # Criar um novo DataFrame com os dados processados
    df_expanded = pd.DataFrame(rows)
    df_expanded['Class'] = None
    df_expanded['Other For'] = None
    
    df_expanded.drop_duplicates(inplace=True)
    df_expanded.reset_index(inplace=True, drop=True)
    
    df_expanded.to_csv(f'./DIET_SDD/{study_group}_{nome}_codebook.csv', sep=',', encoding='utf-8')
    
    dictionary_mapping = df_now_total[['Variable_Name']]
    dictionary_mapping = dictionary_mapping.drop_duplicates(subset='Variable_Name', keep='first')
    dictionary_mapping['Attribute'] = None
    dictionary_mapping['attributeOf'] = None
    dictionary_mapping['Unit'] = None
    dictionary_mapping['Time'] = None
    dictionary_mapping['Entity'] = None
    dictionary_mapping['Role'] = None
    dictionary_mapping['Relation'] = None
    dictionary_mapping['inRelationTo'] = None
    dictionary_mapping['wasDerivedFrom'] = None
    dictionary_mapping['wasGeneratedBy'] = None
    dictionary_mapping.reset_index(inplace=True, drop=True)
    dictionary_mapping.rename(columns={"Variable_Name": "Column"}, inplace=True)
    dictionary_mapping.reset_index(inplace=True, drop=True)
    dictionary_mapping.to_csv(f'./DIET_SDD/{study_group}_{nome}_dictionary_mapping.csv', sep=',', encoding='utf-8')
    
    
    
    
    # display(df_expanded)
    # display(dictionary_mapping)
    # break